# Task #2: Reshape QPASS observations and check data quality

This notebook now uses Dylan's task #1 loader for all workbooks in `data/raw/QPASS/`, then calls the task #2 functions in `src/qpass_observations.py`.

Run cells in order. If the raw folder is empty, first run `python -m src.download_qpass` from the repository root. Original workbooks are never edited. Loader exclusions are retained in metadata; this audit covers rows returned by the loader.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.qpass_loader import load_all_qpass
from src.qpass_observations import (
    reshape_loaded_observations,
    validate_observations,
)

RAW_DIR = REPO_ROOT / "data" / "raw" / "QPASS"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "qpass-notebook"


## 1. Load the supported workbooks

Dylan's loader combines the supported workbook formats into one DataFrame, preserving source file, sheet, row, academic year, and quarter. Counts remain in `open_8am`, `open_10am`, `open_12pm`, and `open_2pm`. Display its audit information as well: rows excluded by the loader do not reach task #2's checks.


In [ ]:
loaded_data = load_all_qpass(RAW_DIR)
loader_audit = loaded_data.attrs.copy()
print("Workbooks loaded:", len(loader_audit["files"]))
print("Source rows returned by loader:", len(loaded_data))
display(loaded_data.head())
print("Loader exclusions (per file and sheet):")
display(loader_audit["dropped_rows"])


## 2. Reshape and standardize values

`reshape_loaded_observations()` uses the shared loader's names. `location` is the facility, `type` is the category, and `sub_type` is the optional subtype. `report_type` is auxiliary metadata, not a replacement for `type`.

Each source row produces four time observations. Original identity fields remain unchanged; `raw_date`, `raw_total_stall_count`, and `raw_open_spaces` retain values before parsing. `date` and `observation_datetime` use local wall time; `total_stall_count` is the workbook count, not verified inventory capacity. See `docs/qpass_observation_contract.md` for downstream integration.


In [ ]:
# These functions are imported from src/qpass_observations.py above.
print("Reshaping function comes from:", reshape_loaded_observations.__module__)


## 3. Define the quality checks

Checks cover missing required identity fields, blank readings, unexpected nonnumeric text, negative/nonfinite/fractional counts, zero capacity, and open spaces above capacity. Whitespace-only readings count as blank. Text such as `FULL`, `n/a` or `-` is preserved and flagged as unexpected text; no numeric meaning is guessed.

Older Format A sheets never had `Report Type` or `Sub Type`. Their status is `not_in_format`, which does not fail required-field checks. Blanks in optional fields of other layouts are marked `blank_or_unavailable` because the loader cannot distinguish an absent optional header from a blank cell. `type` remains required in every format.

`exact_repeated_record` means the identity and capacity/count payload repeat. `conflicting_observation` means an identity has differing payloads. A row can have both flags when an exact repeated subset occurs inside a conflicting group. All records remain; `observation_group` groups related identities for review. Identity includes datetime, location, type, sub_type, and level, but excludes source-row provenance so repeats across sheets/files are detected.

Per-check counts overlap. `any_check` counts unique failed observations.


The validation function is imported above rather than redefined here. Open `src/qpass_observations.py` to read its implementation.

`validate_observations(observations)` returns two DataFrames: the observations with flags, and per-check pass/fail counts. Tests in `tests/test_qpass_observations.py` call this same function with small sample tables containing known problems.


## 4. Run reshaping and validation

The adapter reshapes every row returned by Dylan's loader into four time observations. Missing counts and partially filled observations are retained and flagged. Rows excluded by the loader are recorded separately in its audit metadata.


The connected workflow is:

```text
Excel workbooks -> load_all_qpass() [Dylan]
               -> reshape_loaded_observations() [Vivaan]
               -> validate_observations() -> results and exports
```

All workbook parsing uses Dylan's shared loader; the separate representative reader has been removed. Source file, sheet_name, source_row, format_version, year, quarter, level and subtype are retained. Loader exclusions remain in audit metadata and should be reviewed separately.


In [ ]:
observations = reshape_loaded_observations(loaded_data)
audited, quality_summary = validate_observations(observations)
questionable_records = audited.loc[~audited["quality_pass"]].copy()

assert len(observations) == 4 * len(loaded_data)
assert len(audited) == len(observations)
print(f"Observation rows: {len(audited):,}")
print(f"Passing rows: {audited['quality_pass'].sum():,}")
print(f"Flagged rows: {len(questionable_records):,}")
display(quality_summary)
display(audited[["source_file", "sheet_name", "source_row", "observation_datetime",
                 "location", "type", "sub_type",
                 "total_stall_count", "open_spaces", "quality_flag"]].head(12))


## 5. Examine questionable records

These records need review, not automatic deletion. Use the source file, sheet, and row to investigate the original values. The sample below displays up to five records per failed check; the full dataframe remains available as `questionable_records`.


In [ ]:
audit_columns = ["source_file", "sheet_name", "source_row", "observation_group",
                 "observation_datetime", "location", "type", "sub_type", "level",
                 "raw_total_stall_count", "raw_open_spaces", "quality_flag"]
for check in quality_summary.loc[quality_summary["failed"].gt(0), "check"]:
    if check == "any_check":
        continue
    print(check)
    matching = audited["quality_flag"].str.split("|").map(lambda flags: check in flags)
    display(audited.loc[matching, audit_columns].head())

print("Optional source-field status (not failed observations):")
display(audited.groupby(["format_version", "report_type_status"], dropna=False).size())


## 6. Export the deliverables

This cell writes generated CSV reports locally under `data/processed/qpass-notebook`. It does not edit the workbook or publish to GitHub. CSV readers should parse `observation_datetime` and `observation_date` as dates when reloading.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
audited.to_csv(OUTPUT_DIR / "observations.csv", index=False)
questionable_records.to_csv(OUTPUT_DIR / "questionable_records.csv", index=False)
quality_summary.to_csv(OUTPUT_DIR / "quality_summary.csv", index=False)
run_metadata = {
    "source_files": loader_audit["files"],
    "loader_audit": loader_audit,
    "observation_rows": len(audited),
    "passing_rows": int(audited["quality_pass"].sum()),
    "flagged_rows": len(questionable_records),
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print("Reports saved to:", OUTPUT_DIR.resolve())


## Quality-flag definitions and unresolved questions

| Flag | Meaning |
| --- | --- |
| `missing_<identity>` | Required datetime, location or type is blank/unparseable |
| `missing_<count>` | Original reading is blank, including whitespace-only cells |
| `unexpected_text_<count>` | Nonblank reading cannot be parsed numerically |
| `negative_<count>` | Count below zero, with original precision retained |
| `nonfinite_<count>` | Infinite numeric value |
| `fractional_<count>` | Finite count is not an integer |
| `zero_capacity` | Workbook total_stall_count is zero |
| `open_exceeds_capacity` | Open count exceeds workbook total_stall_count |
| `exact_repeated_record` | Same complete identity and capacity/count payload |
| `conflicting_observation` | Same complete identity, different capacity/count payload |
| `pass` | No implemented checks fail |

- Report Type is optional metadata; type is the category. No category mapping is guessed.
- Optional fields marked not_in_format are structurally absent, not failed required values.
- Review exact repeats and conflicting groups before any deduplication or aggregation.
- Verify capacities with historical inventory; missing/text counts are not zero.
- Extremely small negative numbers may reflect floating-point calculations. Values remain unchanged pending review.
- This audit does not verify permits, entrances, or forecast reliability.
- The contract matches the loader names consumed by the remote label/capacity code; Daira and Hanrong still need to confirm their preferred integration entry points. See the contract document.


## Combined audit results - 2026-10-18 (PR #9 revisions)

Generated with the shared loader and revised task #2 rules. This is a dated snapshot; rerun and refresh it when inputs or code change.

- Workbooks: **15**.
- Loader rows: **23,428**.
- Timed observations: **93,712** (four per loader row).
- Passing: **77,336**.
- Flagged: **16,376**.

The previous snapshot incorrectly required Report Type for older formats. It is now optional metadata; type is the required category. These counts supersede that snapshot. All flagged records remain in observations.csv.

| Check | Failed | Passed |
| --- | ---: | ---: |
| `missing_observation_datetime` | 0 | 93,712 |
| `missing_location` | 0 | 93,712 |
| `missing_type` | 252 | 93,460 |
| `missing_total_stall_count` | 72 | 93,640 |
| `unexpected_text_total_stall_count` | 0 | 93,712 |
| `negative_total_stall_count` | 0 | 93,712 |
| `nonfinite_total_stall_count` | 0 | 93,712 |
| `fractional_total_stall_count` | 0 | 93,712 |
| `missing_open_spaces` | 5,376 | 88,336 |
| `unexpected_text_open_spaces` | 5 | 93,707 |
| `negative_open_spaces` | 1 | 93,711 |
| `nonfinite_open_spaces` | 0 | 93,712 |
| `fractional_open_spaces` | 60 | 93,652 |
| `zero_capacity` | 8,052 | 85,660 |
| `open_exceeds_capacity` | 5,442 | 88,270 |
| `exact_repeated_record` | 80 | 93,632 |
| `conflicting_observation` | 3,304 | 90,408 |
| `any_check` | 16,376 | 77,336 |

Flags overlap. any_check counts unique failing observations.

### Optional fields absent from Format A

These fields did not exist in the old layout and do not fail required-value checks.

| Format | Report Type status | Observations |
| --- | --- | ---: |
| A_wide_weekly | not_in_format | 40,604 |
| B_long_weekly | blank_or_unavailable | 20 |
| B_long_weekly | present | 31,792 |
| C_long_single | present | 21,296 |

Format A: **40,604** observations; **37,709** pass all implemented checks. Missing report_type alone does not reject them.

### Actual flagged examples

Source references identify the original worksheet row. Explanations are reasons for review, not automatic corrections.

**missing_type**

- Source: `QPASS 2018-2019-1 Summer.xlsx` -> `3 Summer 2018 July 2nd` -> Excel row 65.
- Identity: 2018-07-02 08:00:00, location `P017`, type `(blank)`, sub_type `(blank)`, level `(blank)`.
- Raw total_stall_count: `258`; raw open_spaces: `(blank)`.

The required identity or original reading is blank; whitespace-only readings are blank too.

**missing_total_stall_count**

- Source: `QPASS 2024-2025-1  Summer.xlsm` -> `QPASS Data Entry` -> Excel row 107.
- Identity: 2024-07-01 08:00:00, location `Campus Point East`, type `(blank)`, sub_type `(blank)`, level `(blank)`.
- Raw total_stall_count: `(blank)`; raw open_spaces: `(blank)`.

The required identity or original reading is blank; whitespace-only readings are blank too.

**missing_open_spaces**

- Source: `QPASS 2018-2019-1 Summer.xlsx` -> `1 Summer 2018 June 18` -> Excel row 43.
- Identity: 2018-06-18 08:00:00, location `Pangea`, type `B`, sub_type `(blank)`, level `Level 3`.
- Raw total_stall_count: `159`; raw open_spaces: `(blank)`.

The required identity or original reading is blank; whitespace-only readings are blank too.

**unexpected_text_open_spaces**

- Source: `QPASS 2019-2020-2 Fall.xlsx` -> `Fall19 Week 1 Data` -> Excel row 44.
- Identity: 2019-09-30 08:00:00, location `Campus Point East`, type `B`, sub_type `(blank)`, level `(blank)`.
- Raw total_stall_count: `242`; raw open_spaces: `-`.

Nonblank text cannot be interpreted numerically under the current rules. It is not a blank or a guessed zero.

**negative_open_spaces**

- Source: `QPASS 2018-2019-3 Winter.xlsx` -> `W19 W1 Data` -> Excel row 61.
- Identity: 2019-01-11 10:00:00, location `Osler`, type `B`, sub_type `(blank)`, level `(blank)`.
- Raw total_stall_count: `552`; raw open_spaces: `-3.064215547965432e-14`.

The strict check finds a negative value. A near-zero value may reflect floating-point arithmetic; no rounding or correction is assumed.

**fractional_open_spaces**

- Source: `QPASS 2018-2019-3 Winter.xlsx` -> `W19 W1 Data` -> Excel row 18.
- Identity: 2019-01-07 08:00:00, location `Osler`, type `A`, sub_type `(blank)`, level `(blank)`.
- Raw total_stall_count: `156`; raw open_spaces: `109.19999999999999`.

The recorded count fails the named logical/numeric check. Verify it against the source and historical capacity before changing it.

**zero_capacity**

- Source: `QPASS 2024-2025-2 Fall.xlsm` -> `QPASS Data Entry (Week 0)` -> Excel row 90.
- Identity: 2024-09-26 08:00:00, location `P602`, type `2 hour A`, sub_type `(blank)`, level `(blank)`.
- Raw total_stall_count: `0`; raw open_spaces: `0`.

The recorded count fails the named logical/numeric check. Verify it against the source and historical capacity before changing it.

**open_exceeds_capacity**

- Source: `QPASS 2018-2019-1 Summer.xlsx` -> `1 Summer 2018 June 18` -> Excel row 45.
- Identity: 2018-06-18 08:00:00, location `Pangea`, type `B`, sub_type `(blank)`, level `Level 4`.
- Raw total_stall_count: `77`; raw open_spaces: `102`.

The recorded count fails the named logical/numeric check. Verify it against the source and historical capacity before changing it.

**exact_repeated_record**

- Source: `QPASS 2024-2025-2 Fall.xlsm` -> `QPASS Data Entry (Week 0)` -> Excel row 110.
- Identity: 2024-09-26 08:00:00, location `Theatre District Garage`, type `S`, sub_type `(blank)`, level `(blank)`.
- Raw total_stall_count: `50`; raw open_spaces: `(blank)`.

Identity and capacity/count payload repeat. Every copy and its provenance are retained.

**conflicting_observation**

- Source: `QPASS 2019-2020-1 Summer.xlsx` -> `SS19 W2 Data` -> Excel row 100.
- Identity: 2019-07-08 08:00:00, location `Pangea`, type `V`, sub_type `(blank)`, level `(blank)`.
- Raw total_stall_count: `145`; raw open_spaces: `136`.

The same identity has differing capacity/count payloads. All versions remain for review.

### Remaining coordination and assumptions

- Confirm the documented shared-name contract with Daira and Hanrong in PR review; their remote code was inspected, but no conversation or messages were sent.
- Both exact repeats and conflicts remain available; do not delete or aggregate either automatically.
- type is category; sub_type is subtype. No values are copied from one field to the other.
- Optional blanks in formats B/C remain blank_or_unavailable because the loader does not expose optional-header presence.
- Review loader exclusions separately. This audit only checks returned rows; run_metadata.json retains loader exclusion metadata.
- Verify inventory capacity and allocation scope before computing a model-ready availability fraction.
- See docs/qpass_observation_contract.md for schema, duplicate semantics and downstream integration.

### Reproduce

Install requirements.txt, run `python -m src.download_qpass`, then restart the notebook kernel and Run All. Exports are written to data/processed/qpass-notebook/ and stay ignored by Git. Run `python -m pytest -q` to check the code.
